<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/hard/seq-seq/seq-to-seq-with-Attention.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("seq-seq", ...)`


# Problem: Write Sequence-to-Sequence with Attention

### Problem Statement
Implement a **Sequence-to-Sequence (Seq2Seq) model with Attention** by completing the required sections. The model consists of an **Encoder** that processes input sequences and a **Decoder** with an attention mechanism that generates output sequences one token at a time. The demo task is **sequence reversal** — a classic task where attention matters, because output step *t* must look up source position *S − 1 − t*.

### Requirements

1. **Encoder Class**:
   - **Layers**:
     - Use an embedding layer to map input tokens to dense vectors.
     - Use an LSTM layer (`batch_first=True`) to capture temporal dependencies in the sequence.
   - **Forward Pass**:
     - Pass the input sequence through the embedding layer.
     - Feed the embedded sequence into the LSTM.
     - Return the LSTM outputs `(batch, src_len, hidden_dim)` and the final hidden and cell states `(num_layers, batch, hidden_dim)`.

2. **Decoder with Attention** (`Decoder(output_dim, embed_dim, hidden_dim, num_layers)`, one decoding step per call):
   - **Layers**:
     - Use an embedding layer to process output sequence tokens.
     - Implement **Bahdanau (additive) attention** between the decoder state and *every* encoder output $h_i$. With $s$ the top-layer decoder hidden state from the previous step (`hidden[-1]`):
       $$
       e_i = v^\top \tanh(W_s\, s + W_h\, h_i), \qquad \alpha = \operatorname{softmax}_i(e), \qquad c = \sum_i \alpha_i h_i
       $$
       The weights must come from comparing $s$ with the encoder outputs' *content*, so the decoder works for any source length. (Predicting the weights from the decoder state alone with a `Linear(..., src_len)` layer is not attention over the source: it ignores what the encoder outputs contain and only works for one fixed length.)
     - Use an LSTM layer to advance the decoder state using the context vector and the embedded input.
     - Use a fully connected output layer to predict the next token.
   - **Forward Pass**:
     - Process the input token through the embedding layer.
     - Compute attention weights using the decoder hidden state and encoder outputs (softmax over source positions).
     - Calculate the context vector by applying the attention weights to the encoder outputs.
     - Combine the context vector with the embedded input (concatenate, project back to `embed_dim`, `tanh`).
     - Feed the combined representation into the LSTM for one step, starting from `(hidden, cell)`.
     - Pass the LSTM output through a fully connected layer to predict the next token.

3. **Training and inference** (provided):
   - **Teacher forcing**: at step $t$ the decoder input is the *ground-truth* previous token $y_{t-1}$ (a reserved `<sos>` token at $t = 0$) and the target is $y_t$.
   - At inference there is no ground truth, so the decoder is fed its own previous prediction (greedy decoding). The mismatch between the two regimes is called *exposure bias*.

*Luong (multiplicative) attention is the common alternative: score $s_t^\top W h_i$ (or a plain dot product) computed from the decoder state* after *the RNN step, with the context combined into the output layer instead of the RNN input.*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
#TODO: Implement the Encoder class
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers):
        super(Encoder, self).__init__()
        # Add layers for embedding and LSTM

    def forward(self, x):
        # Define the forward pass for the encoder
        ...

# Define the Decoder with Attention
#TODO: Implement the Decoder class with Attention
class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim, num_layers):
        super(Decoder, self).__init__()
        # Add layers for embedding, additive attention (W_s, W_h, v), combining, LSTM, and output

    def forward(self, x, encoder_outputs, hidden, cell):
        # x: (batch,) previous target tokens
        # encoder_outputs: (batch, src_len, hidden_dim); hidden, cell: (num_layers, batch, hidden_dim)
        embedded = ...

        # Attention mechanism: score every encoder output against hidden[-1]
        attention_weights = ...  # (batch, src_len), softmax over the source positions
        context_vector = ...     # (batch, 1, hidden_dim)

        # Combine context and embedded input
        combined = torch.cat(...)
        combined = torch.tanh(...).unsqueeze(1)

        # LSTM and output
        lstm_out, (hidden, cell) = self.lstm(...)
        output = self.fc_out(lstm_out.squeeze(1))
        return output, hidden, cell

In [ ]:
# Synthetic task: reverse the source sequence
torch.manual_seed(42)
src_vocab_size = 20
tgt_vocab_size = 20
SOS = 0                      # token 0 is reserved as <sos>; real tokens are 1..19
src_seq_length = 10
tgt_seq_length = src_seq_length
batch_size = 64

def make_batch(batch_size):
    src = torch.randint(1, src_vocab_size, (batch_size, src_seq_length))
    return src, src.flip(1)  # target = reversed source

# Initialize models, loss function, and optimizer
input_dim = src_vocab_size
output_dim = tgt_vocab_size
embed_dim = 32
hidden_dim = 64
num_layers = 2

encoder = Encoder(input_dim, embed_dim, hidden_dim, num_layers)
decoder = Decoder(output_dim, embed_dim, hidden_dim, num_layers)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.003)

In [ ]:
# Training loop (teacher forcing), a fresh random batch every step
steps = 300
for step in range(steps):
    src_data, tgt_data = make_batch(batch_size)
    encoder_outputs, (hidden, cell) = encoder(src_data)
    loss = 0
    decoder_input = torch.full((batch_size,), SOS, dtype=torch.long)  # <sos>

    for t in range(tgt_seq_length):
        output, hidden, cell = decoder(decoder_input, encoder_outputs, hidden, cell)
        loss += criterion(output, tgt_data[:, t])
        decoder_input = tgt_data[:, t]  # Teacher forcing: feed the ground-truth token
    loss = loss / tgt_seq_length        # mean cross-entropy per target token

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 50 steps
    if (step + 1) % 50 == 0:
        print(f"Step [{step + 1}/{steps}] - Loss: {loss.item():.4f}")

In [ ]:
# Test the sequence-to-sequence model with new input (greedy decoding)
test_input, test_target = make_batch(1)
with torch.no_grad():
    encoder_outputs, (hidden, cell) = encoder(test_input)
    decoder_input = torch.full((1,), SOS, dtype=torch.long)  # <sos>
    output_sequence = []

    for _ in range(tgt_seq_length):
        output, hidden, cell = decoder(decoder_input, encoder_outputs, hidden, cell)
        predicted = output.argmax(1)
        output_sequence.append(predicted.item())
        decoder_input = predicted  # feed back the model's own prediction

    print(f"Input:    {test_input[0].tolist()}")
    print(f"Expected: {test_target[0].tolist()}")
    print(f"Output:   {output_sequence}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("seq-seq", Encoder=Encoder, Decoder=Decoder)